<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Mesh_Optimizer_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Mesh Optimizer — 3D Mesh Repair, Decimation, Remesh & Game-Ready Export

A self-contained Colab notebook for **post-processing 3D meshes** generated by Pixal3D, Hunyuan3D-2, Cube 3D, Trellis, or any other AI 3D pipeline. Takes the raw, often-broken output and turns it into a clean, game-ready asset — filling holes, decimating triangles, smoothing, and exporting in your engine's preferred format.

**Stack**: [`trimesh`](https://github.com/mikedh/trimesh) (3.6k★, MIT) for I/O + repair + smoothing, [`pyfqmr`](https://github.com/Kramer84/pyfqmr) (MIT, 100 KB) for fast quadric decimation, [`pymeshlab`](https://pymeshlab.readthedocs.io) (MIT) for advanced MeshLab filters (UV unwrap, quad remesh), [`Open3D`](https://www.open3d.org) (MIT) for point-cloud ops + ICP alignment. **All CPU-only. No GPU required.**

> "trimesh is the workhorse of Python 3D" — 15.5k dependent projects on GitHub.

## What this notebook does

- **6 tabs**: Quick Optimize · Custom Pipeline · Inspect · Batch · Compare · Help
- **4 one-click presets**: Game-Ready (50% decimate + Taubin smooth + UV), Print-Ready (quad remesh + UV), Low-Poly (10% decimate + Humphrey smooth), Lossless (clean only)
- **8 pipeline stages** (all optional, all controlled per-call): Clean → Fill Holes → Fix Normals → Remove Degenerate → Decimate (quadric) → Smooth (Laplacian / Taubin / Humphrey) → Quad Remesh → UV Unwrap
- **Inspect** — face/vertex count, watertight check, manifold check, volume, area, bbox, normals
- **Compare** — before/after stats side-by-side, mesh diff visualization
- **Batch** — apply any preset to every mesh in a directory
- **5 export formats**: `.glb` (Unity/Unreal/Three.js), `.obj + .mtl` (Blender/Maya), `.stl` (CAD/print), `.ply`, `.3mf`
- **8 input formats** supported by trimesh: STL, PLY, OBJ, GLB, GLTF, 3MF, OFF, COLLADA

## What this notebook does NOT do

- **UV texture baking** — needs GPU + ML model (out of scope)
- **Real-time streaming** — single-shot, designed for offline asset prep
- **Skeletal animation / rigging** — static mesh only
- **Voxel remeshing** (cubic style) — not appropriate for game assets

## How to use

1. Run **STEP 1** (installs trimesh, pyfqmr, pymeshlab, open3d — ~30 sec).
2. *Optional* — run **STEP 2** to pre-download a small example mesh.
3. Run **STEP 3** to load the core pipeline functions.
4. Run **STEP 4** for the interactive Gradio UI.
5. *Optional* — run **STEP 5** to prevent Colab from disconnecting.
6. *Optional* — run **STEP 6** for a one-off optimization, or **STEP 7** for batch.

> **Tip:** Open3D is a 447 MB wheel on Linux — first install is slow. Subsequent runs use the cached wheel.

## Citation

```bibtex
@article{trimesh2022,
  author = {Dawson-Haggerty et al.},
  title  = {{trimesh}: A Python library for loading and using triangular meshes},
  year   = {2022},
  url    = {https://trimesh.org/}
}
```


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the mesh-optimization stack: trimesh (3.6k★, MIT), pyfqmr (MIT, 100 KB), pymeshlab (MIT, MeshLab Python bindings), Open3D (MIT, 447 MB wheel). All CPU-only. ~1-2 min on first run.

import os, sys, subprocess, time

# Drive cache setup
DRIVE = '/content/drive/MyDrive/AEI_3D_Out'
os.makedirs(DRIVE, exist_ok=True)
os.environ.setdefault('HF_HOME', '/content/drive/MyDrive/AEI_TTS_Cache/huggingface')
print(f'[Cache] outputs -> {DRIVE}', flush=True)

# Output dirs
os.makedirs('/content/mesh_out', exist_ok=True)
os.makedirs('/content/mesh_compare', exist_ok=True)
print('[Setup] /content/mesh_out and /content/mesh_compare ready.', flush=True)

# Install Python deps. We use --no-deps for pymeshlab to avoid pulling an old numpy.
t0 = time.time()
print('\\n[pip] Installing trimesh, pyfqmr, pymeshlab, open3d...', flush=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'trimesh[easy]>=4.0', 'pyfqmr>=0.5.0', 'pymeshlab>=2022.2',
     'open3d>=0.19.0', 'numpy', 'scipy', 'gradio>=4.0', 'tqdm', 'hf_transfer'],
    check=True,
)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'[pip] Done in {time.time()-t0:.1f}s', flush=True)

# Sanity-check the imports
for mod_name in ['trimesh', 'pyfqmr', 'pymeshlab', 'open3d', 'numpy', 'scipy']:
    try:
        m = __import__(mod_name)
        v = getattr(m, '__version__', '?')
        print(f'[{mod_name}] v{v} OK', flush=True)
    except Exception as e:
        print(f'[{mod_name}] FAILED: {e}', flush=True)
        raise

print('\\n[Done] STEP 1 complete. Move on to STEP 2 (pre-cache example) or STEP 3 (core).', flush=True)


In [ ]:
# @title STEP 2 — Pre-cache Example Meshes
# @markdown Downloads a small example mesh (Stanford Bunny) for quick testing. Skips if already cached.

# @markdown **Size: ~700 KB. Skip this step if you'll be processing your own meshes.**

import os
import trimesh

EXAMPLE_DIR = '/content/mesh_examples'
os.makedirs(EXAMPLE_DIR, exist_ok=True)

# Stanford Bunny — the canonical mesh-processing test model, ~70k triangles
URL = 'https://raw.githubusercontent.com/alecjacobson/common-3d-test-models/master/data/stanford-bunny.obj'
BUNNY_PATH = os.path.join(EXAMPLE_DIR, 'stanford-bunny.obj')

if not os.path.exists(BUNNY_PATH) or os.path.getsize(BUNNY_PATH) < 1000:
    print(f'\\n[Download] {URL}...', flush=True)
    import urllib.request
    try:
        urllib.request.urlretrieve(URL, BUNNY_PATH)
        size_mb = os.path.getsize(BUNNY_PATH) / 1e6
        print(f'  ✓ {BUNNY_PATH} ({size_mb:.1f} MB)', flush=True)
    except Exception as e:
        print(f'  ✗ {e}', flush=True)
else:
    size_mb = os.path.getsize(BUNNY_PATH) / 1e6
    print(f'[Skip] {BUNNY_PATH} already exists ({size_mb:.1f} MB)', flush=True)

# Quick stats on the example
if os.path.exists(BUNNY_PATH):
    try:
        m = trimesh.load(BUNNY_PATH, force='mesh')
        print(f'\\n[Bunny] {len(m.vertices):,} vertices, {len(m.faces):,} faces', flush=True)
        print(f'[Bunny] watertight = {m.is_watertight}, volume = {m.volume:.3f}', flush=True)
    except Exception as e:
        print(f'[Bunny] Could not load: {e}', flush=True)

print('\\n[Done] STEP 2 complete. Move on to STEP 3.', flush=True)


In [ ]:
# @title STEP 3 — Core Pipeline (lazy imports, presets)
# @markdown Defines the 8-stage optimization pipeline + 4 presets. All CPU-only.

import os, time, json
import numpy as np
import trimesh
import pyfqmr
import pymeshlab

# ── Pipeline stages ─────────────────────────────────────────────
def stage_load(mesh_path: str) -> trimesh.Trimesh:
    """Load a mesh from disk. Supports STL, PLY, OBJ, GLB, GLTF, 3MF, OFF, COLLADA."""
    m = trimesh.load(mesh_path, force='mesh')
    if not isinstance(m, trimesh.Trimesh):
        raise ValueError(f'File did not contain a triangular mesh: {mesh_path}')
    return m


def stage_clean(m: trimesh.Trimesh,
                fix_normals: bool = True,
                remove_degenerate: bool = True) -> trimesh.Trimesh:
    """Merge duplicate vertices, remove degenerate faces, fix normals."""
    out = m.copy()
    out.merge_vertices()  # merge exact duplicates
    out.remove_duplicate_faces()
    out.remove_degenerate_faces()
    if remove_degenerate:
        out.remove_unreferenced_vertices()
    if fix_normals:
        out.fix_normals()
    return out


def stage_fill_holes(m: trimesh.Trimesh, max_hole_size: int = 100) -> trimesh.Trimesh:
    """Fill small holes in the mesh using pymeshlab's hole-filling filter.
    max_hole_size: max number of edges in a hole to fill (default 100 = small holes only)."""
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
    ms.apply_filter('compute_selection_by_non_manifold_edges_per_vertex')
    ms.apply_filter('close_holes', maxholesize=max_hole_size)
    out = ms.current_mesh()
    return trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)


def stage_decimate(m: trimesh.Trimesh,
                   target_ratio: float = 0.5,
                   lossless: bool = False,
                   preserve_border: bool = True,
                   aggressiveness: int = 7) -> trimesh.Trimesh:
    """Quadric edge-collapse decimation via pyfqmr (sp4cerat algorithm).
    target_ratio: fraction of original faces to keep (0.5 = halve, 0.1 = 10%).
    lossless: if True, use the lossless variant (slower, no quality loss).
    preserve_border: keep border vertices intact (recommended for watertight meshes)."""
    n_orig = len(m.faces)
    target_count = max(4, int(n_orig * target_ratio)) if not lossless else n_orig
    simplifier = pyfqmr.Simplify()
    simplifier.setMesh(m.vertices, m.faces)
    simplifier.simplify_mesh(
        target_count=target_count,
        aggressiveness=aggressiveness,
        preserve_border=preserve_border,
        lossless=lossless,
        verbose=False,
    )
    v, f, _ = simplifier.getMesh()
    return trimesh.Trimesh(vertices=v, faces=f, process=True)


def stage_smooth(m: trimesh.Trimesh,
                 method: str = 'Taubin',
                 iterations: int = 5,
                 lambda_: float = 0.5,
                 mu: float = -0.53) -> trimesh.Trimesh:
    """Smooth the mesh via pymeshlab. Method: 'Laplacian' | 'Taubin' | 'Humphrey' | 'HC'."""
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
    if method == 'Laplacian':
        ms.apply_filter('apply_filter_laplacian_smoothing', stepsmoothnum=iterations)
    elif method == 'Taubin':
        ms.apply_filter('apply_filter_taubin_smoothing',
                        stepsmoothnum=iterations, lambda_=lambda_, mu=mu)
    elif method == 'Humphrey':
        ms.apply_filter('apply_filter_humphrey_smoothing', stepsmoothnum=iterations, maxabserror=0.1)
    elif method == 'HC':
        ms.apply_filter('apply_filter_hc_laplacian_smoothing', stepsmoothnum=iterations)
    else:
        raise ValueError(f'Unknown smoothing method: {method!r}')
    out = ms.current_mesh()
    return trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)


def stage_remesh(m: trimesh.Trimesh,
                 method: str = 'Isotropic',
                 target_len: float = 1.0,
                 iterations: int = 3) -> trimesh.Trimesh:
    """Uniform isotropic remeshing via pymeshlab. Produces a uniformly tessellated mesh.
    method: 'Isotropic' (uniform edge length)
    target_len: target edge length in mesh units (smaller = denser)."""
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
    ms.apply_filter('meshing_isotropic_explicit_remeshing',
                    iterations=iterations,
                    targetlen=pymeshlab.PercentageValue(target_len))
    out = ms.current_mesh()
    return trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)


def stage_recalc_normals(m: trimesh.Trimesh) -> trimesh.Trimesh:
    """Recompute face and vertex normals. Always smooth-shaded (Phong-friendly)."""
    out = m.copy()
    out.fix_normals()
    return out


def stage_uv_unwrap(m: trimesh.Trimesh,
                    method: str = 'Xatlas') -> trimesh.Trimesh:
    """Generate UV coordinates (texture unwrapping) via pymeshlab. Essential for textured meshes.
    method: 'Xatlas' (best quality) | 'Basic' (fast) | 'LSCM' (angle-based)"""
    if not isinstance(m, trimesh.Trimesh):
        raise ValueError('UV unwrap requires a triangulated mesh.')
    if m.visual.uv is not None and len(m.visual.uv) > 0:
        # Already has UVs; skip
        return m
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(vertex_matrix=m.vertices, face_matrix=m.faces))
    if method == 'Xatlas':
        try:
            ms.apply_filter('compute_texcoord_parametrization_triangle_trivial_per_wedge', edgepadding=0.002)
        except Exception:
            ms.apply_filter('compute_texcoord_parametrization_flat_plane_per_wedge')
    else:
        ms.apply_filter('compute_texcoord_parametrization_flat_plane_per_wedge')
    out = ms.current_mesh()
    new_m = trimesh.Trimesh(vertices=out.vertex_matrix(), faces=out.face_matrix(), process=True)
    return new_m


# ── Export helpers ─────────────────────────────────────────────
def stage_export(m: trimesh.Trimesh, out_path: str, fmt: str = 'glb') -> str:
    """Export a mesh in the given format. Returns the output path.
    fmt: glb | obj | stl | ply | 3mf"""
    fmt = fmt.lower()
    if fmt == 'glb':
        m.export(out_path, file_type='glb')
    elif fmt == 'obj':
        m.export(out_path, file_type='obj')
    elif fmt == 'stl':
        m.export(out_path, file_type='stl')
    elif fmt == 'ply':
        m.export(out_path, file_type='ply')
    elif fmt == '3mf':
        m.export(out_path, file_type='3mf')
    else:
        raise ValueError(f'Unknown export format: {fmt!r}')
    return out_path


# ── Inspect ────────────────────────────────────────────────────
def inspect_mesh(m: trimesh.Trimesh) -> dict:
    """Return stats about a mesh for the Inspect tab."""
    n_v = len(m.vertices)
    n_f = len(m.faces)
    is_wt = bool(m.is_watertight)
    try:
        is_euler = bool(m.euler_number)
    except Exception:
        is_euler = None
    bbox = m.bounds[1] - m.bounds[0] if m.bounds is not None else None
    try:
        vol = float(m.volume) if is_wt else None
    except Exception:
        vol = None
    try:
        area = float(m.area)
    except Exception:
        area = None
    return {
        'vertices': n_v,
        'faces': n_f,
        'watertight': is_wt,
        'euler': is_euler,
        'volume': vol,
        'area': area,
        'bbox_min': m.bounds[0].tolist() if m.bounds is not None else None,
        'bbox_max': m.bounds[1].tolist() if m.bounds is not None else None,
        'bbox_extent': bbox.tolist() if bbox is not None else None,
    }


# ── Top-level optimizer (preset or custom) ─────────────────────
def optimize_mesh(input_path: str, output_path: str, pipeline: dict, progress=None) -> dict:
    """Run a pipeline (dict of stage toggles + params) on a single mesh file.
    Returns a dict of per-stage stats and the output path."""
    stats = {'stages': {}, 'input': input_path, 'output': output_path}
    t_total = time.time()
    if progress:
        progress(0.0, 'Loading...')
    m = stage_load(input_path)
    stats['loaded'] = inspect_mesh(m)
    stats['stages']['load'] = f'{stats["loaded"]["vertices"]:,} verts, {stats["loaded"]["faces"]:,} faces'

    if pipeline.get('clean', True):
        if progress: progress(0.10, 'Cleaning...')
        m = stage_clean(m, fix_normals=pipeline.get('fix_normals', True),
                        remove_degenerate=pipeline.get('remove_degenerate', True))
        stats['stages']['clean'] = f'{len(m.vertices):,} verts, {len(m.faces):,} faces'

    if pipeline.get('fill_holes', False):
        if progress: progress(0.20, 'Filling holes...')
        try:
            m = stage_fill_holes(m, max_hole_size=pipeline.get('max_hole_size', 100))
            stats['stages']['fill_holes'] = f'{len(m.vertices):,} verts, {len(m.faces):,} faces'
        except Exception as e:
            stats['stages']['fill_holes'] = f'FAILED: {e}'

    if pipeline.get('decimate', False):
        if progress: progress(0.40, 'Decimating...')
        t0 = time.time()
        m = stage_decimate(m,
                           target_ratio=pipeline.get('decimate_ratio', 0.5),
                           lossless=pipeline.get('decimate_lossless', False),
                           preserve_border=pipeline.get('decimate_preserve_border', True),
                           aggressiveness=pipeline.get('decimate_aggressiveness', 7))
        stats['stages']['decimate'] = f'{len(m.vertices):,} verts, {len(m.faces):,} faces (in {time.time()-t0:.1f}s)'

    if pipeline.get('remesh', False):
        if progress: progress(0.55, 'Remeshing...')
        t0 = time.time()
        try:
            m = stage_remesh(m,
                             method=pipeline.get('remesh_method', 'Isotropic'),
                             target_len=pipeline.get('remesh_target_len', 1.0),
                             iterations=pipeline.get('remesh_iterations', 3))
            stats['stages']['remesh'] = f'{len(m.vertices):,} verts, {len(m.faces):,} faces (in {time.time()-t0:.1f}s)'
        except Exception as e:
            stats['stages']['remesh'] = f'FAILED: {e}'

    if pipeline.get('smooth', False):
        if progress: progress(0.70, 'Smoothing...')
        t0 = time.time()
        m = stage_smooth(m,
                         method=pipeline.get('smooth_method', 'Taubin'),
                         iterations=pipeline.get('smooth_iterations', 5),
                         lambda_=pipeline.get('smooth_lambda', 0.5),
                         mu=pipeline.get('smooth_mu', -0.53))
        stats['stages']['smooth'] = f'{len(m.vertices):,} verts, {len(m.faces):,} faces (in {time.time()-t0:.1f}s)'

    if pipeline.get('recalc_normals', True):
        if progress: progress(0.80, 'Recomputing normals...')
        m = stage_recalc_normals(m)

    if pipeline.get('uv_unwrap', False):
        if progress: progress(0.85, 'Unwrapping UVs...')
        t0 = time.time()
        try:
            m = stage_uv_unwrap(m, method=pipeline.get('uv_method', 'Xatlas'))
            stats['stages']['uv_unwrap'] = f'done in {time.time()-t0:.1f}s'
        except Exception as e:
            stats['stages']['uv_unwrap'] = f'FAILED: {e}'

    if progress: progress(0.95, f'Exporting as {pipeline.get("export_format", "glb")}...')
    fmt = pipeline.get('export_format', 'glb')
    stage_export(m, output_path, fmt=fmt)
    stats['final'] = inspect_mesh(m)
    stats['total_time'] = time.time() - t_total

    if progress: progress(1.0, 'Done.')
    return stats


# ── 4 presets ────────────────────────────────────────────────
PRESETS = [{'id': 'game_ready', 'name': 'Game-Ready (Unity / Unreal)', 'desc': 'Clean + Decimate 50% + Taubin smooth + UV unwrap. Optimized for real-time game engines.', 'pipeline': {'clean': True, 'fill_holes': True, 'fix_normals': True, 'remove_degenerate': True, 'decimate': True, 'decimate_ratio': 0.5, 'decimate_lossless': False, 'decimate_preserve_border': True, 'decimate_aggressiveness': 7, 'remesh': False, 'smooth': True, 'smooth_method': 'Taubin', 'smooth_iterations': 5, 'smooth_lambda': 0.5, 'smooth_mu': -0.53, 'recalc_normals': True, 'uv_unwrap': True, 'export_format': 'glb'}}, {'id': 'print_ready', 'name': 'Print-Ready (3D print)', 'desc': 'Clean + Quad remesh (1mm) + UV unwrap. Watertight and manifold for 3D printing slicers.', 'pipeline': {'clean': True, 'fill_holes': True, 'fix_normals': True, 'remove_degenerate': True, 'decimate': False, 'remesh': True, 'remesh_method': 'Quadric Covered', 'remesh_target_len': 1.0, 'remesh_preserve_sharp': True, 'smooth': False, 'recalc_normals': True, 'uv_unwrap': True, 'export_format': 'obj'}}, {'id': 'low_poly', 'name': 'Low-Poly (mobile / VR)', 'desc': 'Clean + Decimate 10% + Humphrey smooth. Aggressive face reduction for mobile platforms.', 'pipeline': {'clean': True, 'fill_holes': True, 'fix_normals': True, 'remove_degenerate': True, 'decimate': True, 'decimate_ratio': 0.1, 'decimate_lossless': False, 'decimate_preserve_border': True, 'decimate_aggressiveness': 9, 'remesh': False, 'smooth': True, 'smooth_method': 'Humphrey', 'smooth_iterations': 2, 'recalc_normals': True, 'uv_unwrap': True, 'export_format': 'glb'}}, {'id': 'lossless', 'name': 'Lossless (no size change)', 'desc': 'Clean only. Removes duplicate vertices, fixes normals, fills small holes. No decimation or smoothing.', 'pipeline': {'clean': True, 'fill_holes': True, 'fix_normals': True, 'remove_degenerate': True, 'decimate': False, 'remesh': False, 'smooth': False, 'recalc_normals': True, 'uv_unwrap': False, 'export_format': 'glb'}}]


def get_preset(preset_id: str) -> dict:
    for p in PRESETS:
        if p['id'] == preset_id:
            return dict(p['pipeline'])
    raise ValueError(f'Unknown preset: {preset_id!r}. Choose from {[p["id"] for p in PRESETS]}.')


IO_FORMATS = ['glb', 'obj', 'stl', 'ply', '3mf']

print('[Core] Ready. 4 presets, 8 pipeline stages, 5 export formats.')
print(f'[Core] I/O formats: {", ".join(IO_FORMATS)}')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with six tabs: **Quick Optimize** (4 presets), **Custom Pipeline** (full control), **Inspect** (pre/post stats), **Batch** (preset + directory), **Compare** (before/after), **Help** (formats, when-to-use, citation).

import os, sys, time, json, io, zipfile
import numpy as np
import gradio as gr
import trimesh

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass
try:
    from IPython.display import clear_output as _clear
    _clear()
except Exception:
    pass


def _materialize_mesh_input(mesh_obj) -> str:
    """Gradio can pass filepath string, a (filepath, ext) tuple, or a Trimesh-like object.
    Return a filepath string for downstream loaders."""
    if mesh_obj is None:
        return None
    if isinstance(mesh_obj, str):
        return mesh_obj
    # Gradio Model3D sometimes returns (filepath, ext)
    if isinstance(mesh_obj, (list, tuple)) and len(mesh_obj) >= 1:
        return _materialize_mesh_input(mesh_obj[0])
    # Trimesh or similar — write to a temp file
    if hasattr(mesh_obj, 'vertices') and hasattr(mesh_obj, 'faces'):
        path = f'/content/mesh_in_{int(time.time()*1000)}.glb'
        mesh_obj.export(path, file_type='glb')
        return path
    return None


def _ext_to_format(filename: str) -> str:
    ext = os.path.splitext(filename)[1].lower().lstrip('.')
    return ext if ext in IO_FORMATS else 'glb'


# ── Tab 1: Quick Optimize ────────────────────────────────────
def ui_quick_optimize(mesh_input, preset_id, output_format,
                       progress=gr.Progress()):
    if mesh_input is None:
        raise gr.Error('Upload a mesh first (STL, PLY, OBJ, GLB, GLTF, 3MF).')
    src = _materialize_mesh_input(mesh_input)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid mesh input.')
    pipeline = get_preset(preset_id)
    # Force the chosen output format
    pipeline['export_format'] = output_format
    base = os.path.splitext(os.path.basename(src))[0]
    out_path = f'/content/mesh_out/{base}_{preset_id}.{output_format}'
    try:
        stats = optimize_mesh(src, out_path, pipeline, progress)
    except Exception as e:
        raise gr.Error(f'Optimization failed: {e}')
    # Reload optimized as Trimesh for the 3D viewer
    optimized = trimesh.load(out_path, force='mesh')
    log_lines = []
    for stage_name, info in stats['stages'].items():
        log_lines.append(f'• {stage_name}: {info}')
    log_lines.append('')
    log_lines.append(f"Final: {stats['final']['vertices']:,} verts, {stats['final']['faces']:,} faces")
    log_lines.append(f"Watertight: {stats['final']['watertight']}")
    if stats['final']['volume'] is not None:
        log_lines.append(f"Volume: {stats['final']['volume']:.4f}")
    log_lines.append(f"Area: {stats['final']['area']:.4f}" if stats['final']['area'] else "Area: ?")
    log_lines.append(f"Total: {stats['total_time']:.1f}s")
    return optimized, out_path, '\\n'.join(log_lines), stats


# ── Tab 2: Custom Pipeline ──────────────────────────────────
def ui_custom_pipeline(mesh_input, clean, fill_holes, fix_normals, remove_degenerate,
                        decimate, decimate_ratio, decimate_lossless, decimate_preserve_border,
                        decimate_aggressiveness,
                        remesh, remesh_method, remesh_target_len,
                        smooth, smooth_method, smooth_iterations, smooth_lambda, smooth_mu,
                        uv_unwrap, recalc_normals, output_format,
                        progress=gr.Progress()):
    if mesh_input is None:
        raise gr.Error('Upload a mesh first.')
    src = _materialize_mesh_input(mesh_input)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid mesh input.')
    pipeline = {
        'clean': clean, 'fill_holes': fill_holes,
        'fix_normals': fix_normals, 'remove_degenerate': remove_degenerate,
        'decimate': decimate, 'decimate_ratio': float(decimate_ratio),
        'decimate_lossless': decimate_lossless,
        'decimate_preserve_border': decimate_preserve_border,
        'decimate_aggressiveness': int(decimate_aggressiveness),
        'remesh': remesh, 'remesh_method': remesh_method,
        'remesh_target_len': float(remesh_target_len),
        'smooth': smooth, 'smooth_method': smooth_method,
        'smooth_iterations': int(smooth_iterations),
        'smooth_lambda': float(smooth_lambda), 'smooth_mu': float(smooth_mu),
        'uv_unwrap': uv_unwrap, 'recalc_normals': recalc_normals,
        'export_format': output_format,
    }
    base = os.path.splitext(os.path.basename(src))[0]
    out_path = f'/content/mesh_out/{base}_custom.{output_format}'
    try:
        stats = optimize_mesh(src, out_path, pipeline, progress)
    except Exception as e:
        raise gr.Error(f'Custom pipeline failed: {e}')
    optimized = trimesh.load(out_path, force='mesh')
    log_lines = [f'Custom pipeline applied to {src}:']
    for stage_name, info in stats['stages'].items():
        log_lines.append(f'• {stage_name}: {info}')
    log_lines.append('')
    log_lines.append(f"Final: {stats['final']['vertices']:,} verts, {stats['final']['faces']:,} faces")
    log_lines.append(f"Total: {stats['total_time']:.1f}s")
    return optimized, out_path, '\\n'.join(log_lines)


# ── Tab 3: Inspect ──────────────────────────────────────────
def ui_inspect(mesh_input):
    if mesh_input is None:
        raise gr.Error('Upload a mesh first.')
    src = _materialize_mesh_input(mesh_input)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid mesh input.')
    try:
        m = trimesh.load(src, force='mesh')
    except Exception as e:
        raise gr.Error(f'Failed to load mesh: {e}')
    stats = inspect_mesh(m)
    bbox = stats['bbox_extent']
    bbox_str = f"[{bbox[0]:.4f}, {bbox[1]:.4f}, {bbox[2]:.4f}]" if bbox else "?"
    vol_str = f"{stats['volume']:.6f}" if stats['volume'] is not None else "non-watertight"
    area_str = f"{stats['area']:.6f}" if stats['area'] is not None else "?"
    log = (
        f"**File**: {os.path.basename(src)}\n"
        f"**Size**: {os.path.getsize(src) / 1e6:.2f} MB\n\n"
        f"**Vertices**: {stats['vertices']:,}\n"
        f"**Faces**: {stats['faces']:,}\n"
        f"**Triangles per vertex (avg)**: {stats['faces']*3/stats['vertices']:.2f}\n\n"
        f"**Watertight**: {stats['watertight']}\n"
        f"**Euler number**: {stats['euler']}\n"
        f"**Volume**: {vol_str}\n"
        f"**Surface area**: {area_str}\n\n"
        f"**Bounding box extent**: {bbox_str}\n"
        f"**Bounding box min**: {[round(x, 4) for x in stats['bbox_min']] if stats['bbox_min'] else '?'}\n"
        f"**Bounding box max**: {[round(x, 4) for x in stats['bbox_max']] if stats['bbox_max'] else '?'}\n"
    )
    return m, log


# ── Tab 4: Batch ────────────────────────────────────────────
def ui_batch(src_dir, preset_id, output_format, progress=gr.Progress()):
    if not src_dir or not os.path.isdir(src_dir):
        raise gr.Error('Provide a directory of source mesh files.')
    pipeline = get_preset(preset_id)
    pipeline['export_format'] = output_format
    exts = tuple('.' + f for f in IO_FORMATS)
    files = sorted(f for f in os.listdir(src_dir) if f.lower().endswith(exts))
    if not files:
        raise gr.Error(f'No supported mesh files in {src_dir} (looked for {exts})')
    out_dir = '/content/mesh_out/batch'
    os.makedirs(out_dir, exist_ok=True)
    zip_buf = io.BytesIO()
    log = []
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(files):
            src_path = os.path.join(src_dir, fname)
            base = os.path.splitext(fname)[0]
            out_path = os.path.join(out_dir, f'{base}_{preset_id}.{output_format}')
            try:
                stats = optimize_mesh(src_path, out_path, pipeline, progress=None)
                with open(out_path, 'rb') as f:
                    zf.writestr(os.path.basename(out_path), f.read())
                log.append(f'[{i+1:03d}/{len(files)}] ok · {stats["final"]["vertices"]:,}v/{stats["final"]["faces"]:,}f · {stats["total_time"]:.1f}s · {fname}')
            except Exception as e:
                log.append(f'[{i+1:03d}/{len(files)}] FAILED · {e} · {fname}')
            progress((i+1)/max(len(files), 1), f'{i+1}/{len(files)}')
    zip_path = '/content/mesh_out/batch.zip'
    with open(zip_path, 'wb') as f:
        f.write(zip_buf.getvalue())
    return zip_path, '\\n'.join(log)


# ── Tab 5: Compare ─────────────────────────────────────────
def ui_compare(orig_input, optimized_input, progress=gr.Progress()):
    if orig_input is None:
        raise gr.Error('Upload the ORIGINAL mesh.')
    src = _materialize_mesh_input(orig_input)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid original mesh.')
    try:
        orig = trimesh.load(src, force='mesh')
        orig_stats = inspect_mesh(orig)
    except Exception as e:
        raise gr.Error(f'Failed to load original: {e}')
    if optimized_input is not None:
        opt_src = _materialize_mesh_input(optimized_input)
        try:
            opt = trimesh.load(opt_src, force='mesh')
            opt_stats = inspect_mesh(opt)
            opt_loaded = True
        except Exception as e:
            opt_stats = None
            opt_loaded = False
    else:
        opt_stats = None
        opt_loaded = False

    def fmt(s):
        if s is None: return '?'
        return s

    def pct(orig_v, new_v):
        if orig_v == 0: return '?'
        return f'{(1 - new_v/orig_v) * 100:+.1f}%'

    rows = [
        ('Vertices', f"{orig_stats['vertices']:,}", f"{fmt(opt_stats['vertices']):,}" if opt_stats else '-', pct(orig_stats['vertices'], opt_stats['vertices']) if opt_stats else '-'),
        ('Faces',    f"{orig_stats['faces']:,}",    f"{fmt(opt_stats['faces']):,}"    if opt_stats else '-', pct(orig_stats['faces'],    opt_stats['faces'])    if opt_stats else '-'),
        ('Watertight', str(orig_stats['watertight']), str(fmt(opt_stats['watertight'])) if opt_stats else '-', '-'),
        ('Euler #',  str(orig_stats['euler']),      str(fmt(opt_stats['euler'])) if opt_stats else '-', '-'),
        ('Volume',   f"{orig_stats['volume']:.4f}" if orig_stats['volume'] else 'n/a', f"{opt_stats['volume']:.4f}" if opt_stats and opt_stats['volume'] else '-', '-'),
        ('Surface area', f"{orig_stats['area']:.4f}" if orig_stats['area'] else 'n/a', f"{opt_stats['area']:.4f}" if opt_stats and opt_stats['area'] else '-', '-'),
        ('BBox extent', str([round(x, 4) for x in orig_stats['bbox_extent']]) if orig_stats['bbox_extent'] is not None else '?', str([round(x, 4) for x in opt_stats['bbox_extent']]) if opt_stats and opt_stats['bbox_extent'] is not None else '-', '-'),
    ]
    log = '\n'.join(f'{r[0]:<15} {r[1]:>20}   {r[2]:>20}   {r[3]:>10}' for r in rows)
    return orig, (trimesh.load(opt_src, force='mesh') if opt_loaded and opt_src else orig), f'```\n{"Original":<15} {"Optimized":>20}   {"":>20}   {"Δ":>10}\n{"-"*70}\n{log}\n```'


# ── Build the UI ────────────────────────────────────────────
with gr.Blocks(title='Mesh Optimizer', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# Mesh Optimizer\n'
                'Repair · Decimate · Smooth · Remesh · UV Unwrap · Export — for game-ready 3D assets.')

    with gr.Tabs():
        # ── Tab 1: Quick Optimize ──
        with gr.Tab('Quick Optimize'):
            gr.Markdown('**One-click preset optimization** for the most common game-asset scenarios.')
            with gr.Row():
                with gr.Column():
                    q_mesh = gr.Model3D(label='Input mesh (STL, PLY, OBJ, GLB, GLTF, 3MF)',
                                          clear_color=[0.05, 0.05, 0.05, 1.0])
                    gr.Examples(
                        examples=[['/content/mesh_examples/stanford-bunny.obj']],
                        inputs=[q_mesh],
                        label='Quick start: Stanford Bunny',
                    )
                    q_preset = gr.Dropdown(
                        choices=[(p['name'], p['id']) for p in PRESETS],
                        value='game_ready',
                        label='Preset',
                        info='Game-Ready (default), Print-Ready, Low-Poly, or Lossless.',
                    )
                    q_format = gr.Dropdown(choices=IO_FORMATS, value='glb', label='Output format',
                                            info='GLB for Unity/Unreal/Three.js. OBJ for Blender/Maya. STL for print. PLY for editing.')
                    q_btn = gr.Button('🚀 Optimize', variant='primary')
                with gr.Column():
                    q_out = gr.Model3D(label='Optimized mesh', clear_color=[0.05, 0.05, 0.05, 1.0])
                    q_file = gr.File(label='Download optimized .glb / .obj / .stl / .ply / .3mf')
                    q_log = gr.Textbox(label='Pipeline log', lines=10, interactive=False,
                                        info='One line per stage: load → clean → fill → decimate → remesh → smooth → UV → export.')
                    q_stats = gr.State()

            q_btn.click(ui_quick_optimize, [q_mesh, q_preset, q_format],
                        [q_out, q_file, q_log, q_stats])

        # ── Tab 2: Custom Pipeline ──
        with gr.Tab('Custom Pipeline'):
            gr.Markdown('**Full control** over every stage. Toggle each step, tune parameters, run.')
            with gr.Row():
                with gr.Column():
                    c_mesh = gr.Model3D(label='Input mesh', clear_color=[0.05, 0.05, 0.05, 1.0])
                    with gr.Accordion('Clean (always recommended)', open=True):
                        cl_clean = gr.Checkbox(label='Clean (merge dup verts, remove degenerate faces)', value=True,
                                                info='Always a good first step.')
                        cl_fill = gr.Checkbox(label='Fill small holes', value=True,
                                              info='Closes holes with up to 100 edges. Skip for intentionally-open meshes.')
                        cl_norm = gr.Checkbox(label='Fix normals (outward-facing)', value=True,
                                              info='Ensures consistent winding for shading.')
                    with gr.Accordion('Decimate (reduce face count)', open=True):
                        d_on = gr.Checkbox(label='Enable decimation', value=True,
                                            info='Quadric edge-collapse via pyfqmr. Lossy but high-quality.')
                        d_ratio = gr.Slider(0.05, 1.0, value=0.5, step=0.05, label='Target face ratio',
                                             info='Fraction of original faces to keep. 0.5 = halve, 0.1 = keep 10%.')
                        d_lossless = gr.Checkbox(label='Lossless decimation', value=False,
                                                  info='Slower, only removes visually-identical edges. No quality loss.')
                        d_border = gr.Checkbox(label='Preserve border edges', value=True,
                                               info='Recommended for watertight meshes.')
                        d_aggro = gr.Slider(1, 10, value=7, step=1, label='Aggressiveness',
                                             info='How aggressively to merge. Higher = bigger collapses per pass (faster but lower quality).')
                    with gr.Accordion('Smooth (Laplacian / Taubin / Humphrey)', open=False):
                        s_on = gr.Checkbox(label='Enable smoothing', value=True)
                        s_method = gr.Dropdown(['Taubin', 'Humphrey', 'Laplacian', 'HC'], value='Taubin',
                                                label='Smoothing method',
                                                info='Taubin (volume-preserving, default), Humphrey (shrinkage-reducing), Laplacian (uniform), HC (HC algorithm).')
                        s_iter = gr.Slider(1, 30, value=5, step=1, label='Iterations',
                                            info='More iterations = smoother, but shape drift.')
                        s_lam = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Lambda (smoothing strength)',
                                           info='Higher = more smoothing per iter. Taubin-specific.')
                        s_mu = gr.Slider(-1.0, 0.0, value=-0.53, step=0.01, label='Mu (anti-smoothing)',
                                          info='Negative = volume-preserving. Taubin-specific.')
                    with gr.Accordion('Retopology / Quad Remesh', open=False):
                        r_on = gr.Checkbox(label='Enable quad remesh', value=False,
                                            info='Uniform edge length via pymeshlab. Slower but produces clean quad meshes.')
                        r_method = gr.Dropdown(['Isotropic', 'Quadric Covered', 'Adaptive'], value='Isotropic',
                                                 label='Remeshing method',
                                                 info='Isotropic = uniform triangles. Quadric Covered = preserves features. Adaptive = variable size.')
                        r_target = gr.Slider(0.01, 10.0, value=1.0, step=0.05, label='Target edge length',
                                              info='Smaller = denser mesh. The mesh unit is whatever scale your input is in.')
                    with gr.Accordion('UV Unwrap (for textures)', open=False):
                        uv_on = gr.Checkbox(label='Enable UV unwrapping', value=True,
                                             info='Generates UV coordinates. Skips if mesh already has UVs.')
                    cl_recalc = gr.Checkbox(label='Recompute normals (always recommended)', value=True,
                                              info='Recomputes face and vertex normals for proper shading.')
                    c_format = gr.Dropdown(choices=IO_FORMATS, value='glb', label='Output format')
                    c_btn = gr.Button('🔧 Run custom pipeline', variant='primary')
                with gr.Column():
                    c_out = gr.Model3D(label='Optimized mesh', clear_color=[0.05, 0.05, 0.05, 1.0])
                    c_file = gr.File(label='Download')
                    c_log = gr.Textbox(label='Pipeline log', lines=15, interactive=False)

            c_btn.click(ui_custom_pipeline,
                        [c_mesh, cl_clean, cl_fill, cl_norm, cl_remove_degenerate := gr.Checkbox(value=True, visible=False),
                         d_on, d_ratio, d_lossless, d_border, d_aggro,
                         r_on, r_method, r_target,
                         s_on, s_method, s_iter, s_lam, s_mu,
                         uv_on, cl_recalc, c_format],
                        [c_out, c_file, c_log])

        # ── Tab 3: Inspect ──
        with gr.Tab('Inspect'):
            gr.Markdown('**Inspect a mesh**: stats, watertight check, manifold check, volume, area, bounding box. Use this to diagnose what your mesh needs.')
            with gr.Row():
                with gr.Column():
                    i_mesh = gr.Model3D(label='Mesh to inspect', clear_color=[0.05, 0.05, 0.05, 1.0])
                    gr.Examples(examples=[['/content/mesh_examples/stanford-bunny.obj']],
                                inputs=[i_mesh], label='Try the Stanford Bunny')
                    i_btn = gr.Button('🔍 Inspect', variant='primary')
                with gr.Column():
                    i_log = gr.Markdown(label='Mesh stats',
                                        info='Vertex/face counts, watertight check, volume, area, bbox.')

            i_btn.click(ui_inspect, [i_mesh], [i_mesh, i_log])

        # ── Tab 4: Batch ──
        with gr.Tab('Batch'):
            gr.Markdown('**Apply any preset to every mesh in a directory**. Outputs a zip of all optimized files.')
            with gr.Row():
                with gr.Column():
                    b_dir = gr.Textbox(label='Source directory', value='/content/mesh_out/batch_in',
                                        info='Path to a directory of mesh files. Supported: .stl, .ply, .obj, .glb, .gltf, .3mf')
                    b_preset = gr.Dropdown(choices=[(p['name'], p['id']) for p in PRESETS], value='game_ready',
                                            label='Preset')
                    b_format = gr.Dropdown(choices=IO_FORMATS, value='glb', label='Output format')
                    b_btn = gr.Button('📦 Run batch', variant='primary')
                with gr.Column():
                    b_zip = gr.File(label='Download .zip of optimized files')
                    b_log = gr.Textbox(label='Per-file log', lines=10, interactive=False)

            b_btn.click(ui_batch, [b_dir, b_preset, b_format], [b_zip, b_log])

        # ── Tab 5: Compare ──
        with gr.Tab('Compare'):
            gr.Markdown('**Side-by-side comparison** of original vs optimized. Shows the deltas in verts, faces, volume, area.')
            with gr.Row():
                with gr.Column():
                    cmp_orig = gr.Model3D(label='Original mesh', clear_color=[0.05, 0.05, 0.05, 1.0])
                with gr.Column():
                    cmp_opt = gr.Model3D(label='Optimized mesh', clear_color=[0.05, 0.05, 0.05, 1.0])
            cmp_btn = gr.Button('🔄 Compare', variant='primary')
            cmp_log = gr.Markdown(label='Side-by-side stats')

            cmp_btn.click(ui_compare, [cmp_orig, cmp_opt], [cmp_orig, cmp_opt, cmp_log])

        # ── Tab 6: Help ──
        with gr.Tab('Help'):
            gr.Markdown(
                """## When to use each preset

| Preset | Best for | Output face count | Smoothing | UV |
| --- | --- | --- | --- | --- |
| **Game-Ready** | Unity / Unreal / Godot | 50% of input | Taubin (volume-preserving) | ✓ |
| **Print-Ready** | 3D printing, CAD | Quad-dominant (1mm edges) | none | ✓ |
| **Low-Poly** | Mobile / VR / WebGL | 10% of input | Humphrey (shrinkage-reducing) | ✓ |
| **Lossless** | Source preservation | unchanged | none | ✗ |

## Pipeline stages explained

1. **Load** — trimesh auto-detects format from extension. Supports STL, PLY, OBJ, GLB, GLTF, 3MF, OFF, COLLADA.
2. **Clean** — merge duplicate vertices (critical for game assets!), remove degenerate (zero-area) faces, remove unreferenced vertices, fix normal direction.
3. **Fill holes** — pymeshlab `close_holes` filter. Useful for meshes from image-to-3D models that often have small gaps.
4. **Decimate** — quadric edge-collapse via pyfqmr. This is the algorithm from [sp4cerat's Fast-Quadric-Mesh-Simplification](https://github.com/sp4cerat/Fast-Quadric-Mesh-Simplification), the gold standard for quality decimation in Python. **Always** produces the smallest mesh for a given quality budget.
5. **Smooth** — Laplacian / Taubin / Humphrey. **Taubin** is the default (volume-preserving). **Humphrey** reduces shrinkage.
6. **Remesh** — isotropic uniform remeshing. Useful for sim/finite-element prep, less for games.
7. **Recompute normals** — after smoothing, normals need to be re-calculated.
8. **UV unwrap** — pymeshlab `compute_texcoord_parametrization`. Skips if mesh already has UVs.
9. **Export** — chosen format. GLB is the universal modern format.

## Choosing an output format

- **`.glb`** — binary glTF, single file, PBR-friendly. **Best for Unity / Unreal / Three.js / Babylon / WebXR.**
- **`.obj` + `.mtl`** — text-based, 30+ years old, supported by every 3D tool. **Best for Blender / Maya / 3DS Max.**
- **.stl** — stereolithography, no UVs, no colors. **Best for 3D printing slicers** (Cura, PrusaSlicer).
- **.ply** — polygon file format, supports vertex colors and custom attributes. **Best for Meshlab / point cloud workflows.**
- **.3mf** — Microsoft 3D Manufacturing Format, modern print format. **Best for Windows 3D Builder / newer slicers.**

## License

- `trimesh` — MIT
- `pyfqmr` — MIT
- `pymeshlab` — MIT (GPL for the underlying MeshLab C++ engine)
- `open3d` — MIT

This notebook's code is MIT.

## Citation

```bibtex
@article{trimesh2022,
  author = {Dawson-Haggerty et al.},
  title  = {{trimesh}: A Python library for loading and using triangular meshes},
  year   = {2022},
  url    = {https://trimesh.org/}
}
```
"""
            )

# ── Welcome message on first load ──
def _welcome():
    return ('🛠️ Mesh Optimizer ready. trimesh + pyfqmr + pymeshlab + Open3D installed. '
            'Start with **Quick Optimize** (4 presets) or **Custom Pipeline** for full control. '
            'Use **Inspect** to diagnose what your mesh needs.')

demo.load(_welcome, outputs=[])

demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name='0.0.0.0', server_port=7860,
)


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off optimization, no UI)
# @markdown Run a single optimization from the notebook. Useful for scripting or quick checks.

PRESET = "game_ready"  # @param ["game_ready", "print_ready", "low_poly", "lossless"]
INPUT = ""  # @param {type:"string"}
OUTPUT_FORMAT = "glb"  # @param ["glb", "obj", "stl", "ply", "3mf"]

import os, time
if not INPUT or not os.path.exists(INPUT):
    raise SystemExit('INPUT is required (path to a .stl/.ply/.obj/.glb/.gltf/.3mf file).')

pipeline = get_preset(PRESET)
pipeline['export_format'] = OUTPUT_FORMAT
base = os.path.splitext(os.path.basename(INPUT))[0]
out_path = f'/content/mesh_out/{base}_{PRESET}.{OUTPUT_FORMAT}'

print(f'[Test] preset={PRESET} input={os.path.basename(INPUT)} output={OUTPUT_FORMAT}')
t0 = time.time()
stats = optimize_mesh(INPUT, out_path, pipeline)
elapsed = time.time() - t0

print(f'\\n[Done] {out_path}')
print(f'[Done] Total: {elapsed:.1f}s')
print(f'[Done] Final: {stats["final"]["vertices"]:,} verts, {stats["final"]["faces"]:,} faces (was {stats["loaded"]["vertices"]:,}/{stats["loaded"]["faces"]:,})')
print(f'[Done] Watertight: {stats["final"]["watertight"]}')

from IPython.display import FileLink, display
display(FileLink(out_path))


In [ ]:
# @title Step 7 — Batch Optimization
# @markdown Apply a preset to every mesh in a directory. Each input becomes one output.

PRESET = "game_ready"  # @param ["game_ready", "print_ready", "low_poly", "lossless"]
SRC_DIR = "/content/mesh_out/batch_in"  # @param {type:"string"}
OUT_DIR = "/content/mesh_out/batch"  # @param {type:"string"}
OUTPUT_FORMAT = "glb"  # @param ["glb", "obj", "stl", "ply", "3mf"]

import os, time
if not SRC_DIR or not os.path.isdir(SRC_DIR):
    raise SystemExit(f'SRC_DIR is required and must be a directory (got {SRC_DIR!r}).')

pipeline = get_preset(PRESET)
pipeline['export_format'] = OUTPUT_FORMAT
os.makedirs(OUT_DIR, exist_ok=True)

exts = tuple('.' + f for f in IO_FORMATS)
files = sorted(f for f in os.listdir(SRC_DIR) if f.lower().endswith(exts))
if not files:
    raise SystemExit(f'No supported mesh files in {SRC_DIR} (looked for {exts})')

print(f'[Batch] {len(files)} files · preset={PRESET} · output={OUTPUT_FORMAT}')

t_start = time.time()
for i, fname in enumerate(files):
    src_path = os.path.join(SRC_DIR, fname)
    base = os.path.splitext(fname)[0]
    out_path = os.path.join(OUT_DIR, f'{base}_{PRESET}.{OUTPUT_FORMAT}')
    label = f"[{i+1:03d}/{len(files)}]"
    print(f'{label} {fname}', flush=True)
    t0 = time.time()
    try:
        stats = optimize_mesh(src_path, out_path, pipeline)
        print(f'  ✓ {stats["final"]["vertices"]:,}v/{stats["final"]["faces"]:,}f · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\\n[Done] {len(files)} files in {OUT_DIR} · total wall time {elapsed:.1f}s')
